In [8]:
import math
import time

import gradio as gr
import pandas as pd
import plotly.express as px
from datasets import load_dataset, Audio
from sdoml_task1.config import DATA_DIR, MODEL_DIR
from sdoml_task1.modeling.predict import ModelLoader

from sdoml_task1.modeling.load import load_features
from sdoml_task1.modeling.train import run_training, save_run
from sdoml_task1.modeling.optimization import OPTIMIZERS, SCHEDULERS, REGULARIZATIONS, ACTIVATIONS
from sdoml_task1.plots import confusion_matrix_figure, training_metrics_figure


In [9]:
X_train, y_train, X_test, y_test = load_features()
df = pd.read_csv(DATA_DIR / "audio_mnist_metadata.csv")


MAX_LAYERS = 10


def update_slider_visibility(n_layers):
    """Show only the first ``n_layers`` hidden-layer inputs."""
    updates = []
    for i in range(MAX_LAYERS):
        if i < n_layers:
            updates.append(gr.Number(visible=True))
        else:
            updates.append(gr.Number(visible=False))
    return updates


def gradio_train(epochs, lr_text, batch_size, n_layers, activation_function, regularization,
                 reg_param, optimizer_name, scheduler_name, limit_epoch, progress=gr.Progress(), *layer_sizes):
    """Read the values from the interface, train the model and return a summary."""

    def show_epoch(epoch, total_epochs, train_loss, val_loss, val_acc):
        progress(epoch / total_epochs, desc=f"Epoch {epoch}/{total_epochs} - val accuracy {val_acc:.3f}")

    try:
        lr = float(lr_text)
    except ValueError:
        raise gr.Error(f"The learning rate must be a number (e.g. 0.001, got '{lr_text}'.")
    if lr <= 0:
        raise gr.Error("The learning rate must be positive.")

    hidden_sizes = []
    for i in range(int(n_layers)):
        if layer_sizes[i] is None or layer_sizes[i] < 1:
            raise gr.Error(f"Hidden layer {i+1} must have at least 1 neuron.")
        hidden_sizes.append(layer_sizes[i])

    if regularization == "None":
        regularization = None

    if regularization == "dropout":
        if reg_param < 0 or reg_param >= 1:
            raise gr.Error("For dropout, the value is between 0 and 1.")

    progress(0, desc="Starting training...")

    metrics = run_training(X_train, y_train, X_test, y_test, hidden_sizes=hidden_sizes,
                           activation_function=activation_function, regularization=regularization,
                           reg_param=reg_param, optimizer_name=optimizer_name, scheduler_name=scheduler_name,
                           epochs=int(epochs), lr=lr, batch_size=int(batch_size), limit_epoch=int(limit_epoch), on_epoch_end=show_epoch)
    save_run(metrics, MODEL_DIR)

    final_acc = metrics["accuracy_val"][-1]
    best_acc = max(metrics["accuracy_val"])

    summary = (
        f"**Model trained and saved:**   \n"
        f"**Final validation accuracy:** {final_acc:.4f}    \n"
        f"**Best validation accuracy:** {best_acc:.4f}    \n"
        f"**Final validation loss:** {metrics['loss_val'][-1]:.4f}"
    )

    metrics_figure = training_metrics_figure(metrics)
    cm_figure = confusion_matrix_figure(
        metrics["confusion_matrices"][-1], len(metrics["confusion_matrices"])
    )

    return summary, metrics_figure, cm_figure


Features loaded from /home/amets/Documents/MASTER/SDOML/project/SDOML-project1/data/processed/features.pkl


In [10]:
if not (MODEL_DIR / "model.pt").is_file():
    print("No saved model found. Training the default model before starting the UI...")
    startup_metrics = run_training(
        X_train,
        y_train,
        X_test,
        y_test,
        hidden_sizes=(64,),
        activation_function="relu",
        epochs=10,
        lr=1e-3,
        batch_size=64,
    )
    save_run(startup_metrics, MODEL_DIR)
else:
    print(f"Using existing model: {MODEL_DIR / 'model.pt'}")


Using existing model: /home/amets/Documents/MASTER/SDOML/project/SDOML-project1/models/model.pt


In [11]:
# Get unique values for each column for the dropdown filters
def get_filter_options():
    filters = {}
    for col in df.columns:
        # Get unique values, excluding NaN
        unique_vals = df[col].dropna().astype(str).unique().tolist()
        # Sort and add "All" option
        filters[col] = ["All"] + sorted(unique_vals)
    return filters

filter_options = get_filter_options()

def filter_data(digit, speaker_id, gender, accent, age, origin, split):
    """Filter the dataframe based on selected criteria"""
    filtered_df = df.copy()

    # Apply filters (only if not "All" is selected)
    if digit != "All":
        filtered_df = filtered_df[filtered_df["digit"].astype(str) == digit]
    if speaker_id != "All":
        filtered_df = filtered_df[filtered_df["speaker_id"].astype(str) == speaker_id]
    if gender != "All":
        filtered_df = filtered_df[filtered_df["gender"].astype(str) == gender]
    if accent != "All":
        filtered_df = filtered_df[filtered_df["accent"].astype(str) == accent]
    if age != "All":
        filtered_df = filtered_df[filtered_df["age"].astype(str) == age]
    if origin != "All":
        filtered_df = filtered_df[filtered_df["origin"].astype(str) == origin]
    if split != "All":
        filtered_df = filtered_df[filtered_df["split"].astype(str) == split]

    # Return both the table and a count
    count = len(filtered_df)
    return filtered_df, f"Showing {count} entries"

def reset_filters():
    """Reset all filters to 'All'"""
    return "All", "All", "All", "All", "All", "All", "All", df, f"Showing {len(df)} entries"

# Map display names to actual column names
column_mapping = {
    "Digit": "digit",
    "Speaker ID": "speaker_id",
    "Gender": "gender",
    "Accent": "accent",
    "Age": "age",
    "Origin": "origin",
    "Split": "split"
}

def update_plot(digit, speaker_id, gender, accent, age, origin, split, plot_column):
    """Generate interactive plot for selected column based on filtered data"""
    filtered_df = df.copy()


    if digit != "All":
        filtered_df = filtered_df[filtered_df["digit"].astype(str) == digit]
    if speaker_id != "All":
        filtered_df = filtered_df[filtered_df["speaker_id"].astype(str) == speaker_id]
    if gender != "All":
        filtered_df = filtered_df[filtered_df["gender"].astype(str) == gender]
    if accent != "All":
        filtered_df = filtered_df[filtered_df["accent"].astype(str) == accent]
    if age != "All":
        filtered_df = filtered_df[filtered_df["age"].astype(str) == age]
    if origin != "All":
        filtered_df = filtered_df[filtered_df["origin"].astype(str) == origin]
    if split != "All":
        filtered_df = filtered_df[filtered_df["split"].astype(str) == split]

    # Map the display name to actual column name
    actual_column = column_mapping[plot_column]

    # Create value counts
    counts = filtered_df[actual_column].value_counts().reset_index()
    counts.columns = [actual_column, "count"]
    counts = counts.sort_values("count", ascending=False)

    # Create interactive bar chart with better styling
    fig = px.bar(
        counts,
        x=actual_column,
        y="count",
        title=f"Distribution by {plot_column}",
        labels={"count": "Count", actual_column: plot_column},
        color="count",
        color_continuous_scale="Viridis"
    )

    fig.update_layout(
        template="plotly_white",
        hovermode="x unified",
        title_font_size=20,
        xaxis_title_font_size=14,
        yaxis_title_font_size=14,
        height=500,
        showlegend=False,
        margin=dict(l=50, r=50, t=80, b=50),
        coloraxis_showscale=False
    )

    fig.update_traces(
        hovertemplate="<b>%{x}</b><br>Count: %{y}<extra></extra>",
        marker_line_color="white",
        marker_line_width=1
    )

    return fig

### Remember to close the port with .close() or by restarting the kernel. If you don't, it sometimes struggles to re-run the code cause the port is busy.

In [12]:
def create_app():
    with gr.Blocks() as interface:
        gr.Markdown("# Spoken Digit Classification")

        with gr.Tab("Data"):
            gr.Markdown("Explore the Audio MNIST metadata.")
            filters = []
            with gr.Row():
                for column in ["digit", "speaker_id", "gender", "accent", "age", "origin", "split"]:
                    dropdown = gr.Dropdown(choices=filter_options[column], value="All", label=column.replace("_", " ").title())
                    filters.append(dropdown)
            with gr.Row():
                data_table = gr.Dataframe(value=df, interactive=False, label="Data")
                result_text = gr.Textbox(value=f"Showing {len(df)} entries", interactive=False, label="Result Count")
            with gr.Row():
                plot_column = gr.Dropdown(
                    choices=["Digit", "Speaker ID", "Gender", "Accent", "Age", "Origin", "Split"],
                    value="Digit",
                    label="Select Column to Plot",
                )
                plots_output = gr.Plot()
            for filter_elem in filters:
                filter_elem.change(fn=filter_data, inputs=filters, outputs=[data_table, result_text])
                filter_elem.change(fn=update_plot, inputs=filters + [plot_column], outputs=plots_output)
            plot_column.change(fn=update_plot, inputs=filters + [plot_column], outputs=plots_output)

        with gr.Tab("Train"):
            with gr.Row():
                with gr.Column(scale=2):
                    epochs_input = gr.Number(value=10, minimum=1, precision=0, label="Epochs")
                    lr_input = gr.Textbox(value="0.001", label="Learning rate")
                    batch_input = gr.Number(value=64, minimum=1, precision=0, label="Batch size")
                    n_layers_slider = gr.Slider(minimum=1, maximum=MAX_LAYERS, value=1, step=1, label="Number of hidden layers")
                    layer_inputs = [gr.Number(value=64, minimum=1, precision=0, label=f"Layer {i + 1} neurons", visible=(i == 0)) for i in range(MAX_LAYERS)]
                    activation = gr.Radio(ACTIVATIONS, value="relu", label="Activation function")
                    optimizer = gr.Radio(OPTIMIZERS, value="Adam", label="Optimizer")
                    scheduler = gr.Radio(SCHEDULERS, value="None", label="Learning rate scheduler")
                    regularization = gr.Radio(REGULARIZATIONS, value="None", label="Regularization")
                    reg_param_input = gr.Number(value=0.0, minimum=0, label="Regularization strength")
                    patience_input = gr.Number(value=10, minimum=1, precision=0, label="Early stopping patience")
                    train_button = gr.Button("Train", variant="primary")
                with gr.Column(scale=1):
                    summary_output = gr.Markdown("*Results will appear here after training.*")
                    loss_plot = gr.Plot(label="Training metrics")
                    cm_plot = gr.Plot(label="Confusion matrix (last epoch)")
            n_layers_slider.change(fn=update_slider_visibility, inputs=n_layers_slider, outputs=layer_inputs)
            train_inputs = [epochs_input, lr_input, batch_input, n_layers_slider, activation, regularization, reg_param_input, optimizer, scheduler, patience_input, *layer_inputs]
            train_button.click(fn=gradio_train, inputs=train_inputs, outputs=[summary_output, loss_plot, cm_plot])

        with gr.Tab("Model"):
            loader = ModelLoader()
            def predict_digit(audio_path: str) -> dict[str, float]:
                probs = loader.make_prediction(audio_path)
                return {str(digit): float(p) for digit, p in enumerate(probs)}
            gr.Interface(fn=predict_digit, inputs=gr.Audio(type="filepath", label="Upload a spoken digit"), outputs=gr.Label(num_top_classes=10, label="Predicted digit"))

    return interface


In [ ]:
try:
    interface.close()
except:
    pass

# Set to true to publish the app
send_it = False

interface = create_app()
interface.launch(server_port=7861, share=send_it, show_error=True)

Closing server running on port: 7861
